# 01 · Data Cleaning

The raw file is the UCI *Online Retail II* dataset: every transaction from a UK-based online gift
retailer between December 2009 and December 2011. A lot of the customers are wholesalers, which
explains some of the very large orders.

Before any analysis I want a table I can trust. This notebook:

1. combines the two yearly sheets
2. removes duplicate rows and fixes data types
3. separates cancellations and puts a number on how much revenue they cost
4. strips out rows that aren't product sales (postage, fees, manual adjustments, test items)
5. splits off orders with no Customer ID (guest checkouts) so they still count for revenue but not for customer analysis
6. adds the date and revenue columns the later notebooks need

I print the shape after every step so it's clear where the rows go.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_FILE = Path("../data/raw/online_retail_II.xlsx")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)

# Running log of how many rows survive each step, used for the summary at the end.
step_log = []

def record(step, frame, note=""):
    """Print the shape after a cleaning step and keep it for the summary table."""
    step_log.append({"Step": step, "Rows": len(frame), "Columns": frame.shape[1], "Note": note})
    print(f"{step:<48} rows={len(frame):>9,}  cols={frame.shape[1]}")

## 1. Load and combine the two sheets

The workbook has one sheet per trading year. Reading the Excel file takes a minute or two.

In [2]:
sheets = pd.read_excel(RAW_FILE, sheet_name=None)

for name, sheet in sheets.items():
    print(f"{name}: {sheet.shape[0]:,} rows, {sheet.InvoiceDate.min():%d %b %Y} to {sheet.InvoiceDate.max():%d %b %Y}")

df = pd.concat(sheets.values(), ignore_index=True)
df = df.rename(columns={"Customer ID": "CustomerID"})
record("Combined both sheets", df)
df.head()

Year 2009-2010: 525,461 rows, 01 Dec 2009 to 09 Dec 2010
Year 2010-2011: 541,910 rows, 01 Dec 2010 to 09 Dec 2011
Combined both sheets                             rows=1,067,371  cols=8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


The date ranges overlap: the first sheet runs to 9 Dec 2010 and the second one starts on 1 Dec 2010.
Those nine days are in the file twice, so the duplicate check below matters more than usual.

In [3]:
overlap = df[(df.InvoiceDate >= "2010-12-01") & (df.InvoiceDate < "2010-12-10")]
print(f"Rows dated 1-9 Dec 2010: {len(overlap):,}  (of which exact duplicates: {overlap.duplicated().sum():,})")

Rows dated 1-9 Dec 2010: 45,046  (of which exact duplicates: 22,844)


## 2. Missing values and duplicates

In [4]:
missing = pd.DataFrame({
    "Missing": df.isna().sum(),
    "% of rows": (df.isna().mean() * 100).round(2),
})
missing[missing.Missing > 0]

,Missing,% of rows
Description,4382,0.41
CustomerID,243007,22.77


About 23% of rows have no Customer ID. That's too much revenue to throw away, so these rows go into
a separate guest-orders table later instead of being dropped. Missing descriptions only show up on
zero-price rows, and those get removed anyway.

In [5]:
n_dupes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
record("Dropped exact duplicate rows", df, f"{n_dupes:,} duplicates removed")

Dropped exact duplicate rows                     rows=1,033,036  cols=8


## 3. Fix data types

`Invoice` and `StockCode` mix integers and strings (e.g. `85048` vs `79323P`, `489434` vs `C489449`),
so I cast them to clean upper-case strings. `CustomerID` comes in as a float because of the blanks;
pandas' nullable integer type fixes that.

In [6]:
df["Invoice"] = df["Invoice"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].astype("string").str.strip()
df["CustomerID"] = df["CustomerID"].astype("Int64")
df["Country"] = df["Country"].astype(str).str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

record("Fixed dtypes", df)
df.dtypes

Fixed dtypes                                     rows=1,033,036  cols=8


Invoice                object
StockCode              object
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
CustomerID              Int64
Country                object
dtype: object

## 4. Separate cancellations

Invoices starting with **C** are cancellations, stored as negative quantities. There are also six
invoices starting with **A**. These are bookkeeping entries ("Adjust bad debt") and not customer
activity, so they get dropped.

In [7]:
is_cancel = df["Invoice"].str.startswith("C")
is_adjust = df["Invoice"].str.startswith("A")

cancellations = df[is_cancel].copy()
cancellations["Revenue"] = cancellations["Quantity"] * cancellations["Price"]

print(f"Cancellation lines: {is_cancel.sum():,}  across {cancellations.Invoice.nunique():,} invoices")
print(f"Bad-debt adjustment lines: {is_adjust.sum()}")

sales = df[~is_cancel & ~is_adjust].copy()
record("Removed cancellations and A-adjustments", sales,
       f"{is_cancel.sum():,} cancellation lines kept aside, {is_adjust.sum()} adjustments dropped")

Cancellation lines: 19,104  across 8,292 invoices
Bad-debt adjustment lines: 6


Removed cancellations and A-adjustments          rows=1,013,926  cols=8


## 5. Remove non-product rows

Not every stock code is a product. Postage, carriage, Amazon fees, bank charges, manual adjustments,
discounts, samples, gift vouchers and test items all show up as line items. I went through every
stock code that doesn't start with a 5-digit number and kept the real products (the `DCGS…`
dotcom gift-shop range and `SP1002`). Everything else is listed below.

In [8]:
NON_PRODUCT_CODES = {
    "POST", "DOT", "C2", "C3",           # postage / carriage
    "M", "D", "S", "B",                  # manual, discount, samples, bad debt
    "BANK CHARGES", "AMAZONFEE", "CRUK", # fees and commissions
    "PADS",                              # £0.001 placeholder line
}
NON_PRODUCT_PREFIXES = ("ADJUST", "GIFT", "TEST")


def is_non_product(stock_codes: pd.Series) -> pd.Series:
    """True for stock codes that are fees, postage, adjustments or test items."""
    return stock_codes.isin(NON_PRODUCT_CODES) | stock_codes.str.startswith(NON_PRODUCT_PREFIXES)


non_product_mask = is_non_product(sales["StockCode"])
(sales[non_product_mask]
   .assign(Revenue=lambda d: d.Quantity * d.Price)
   .groupby("StockCode")
   .agg(Description=("Description", "first"), Lines=("Invoice", "size"), Revenue=("Revenue", "sum"))
   .sort_values("Lines", ascending=False))

,Description,Lines,Revenue
StockCode,,,
POST,POSTAGE,1858,"125,682.42"
DOT,DOTCOM POSTAGE,1422,"309,854.11"
M,Manual,858,"339,241.29"
C2,CARRIAGE,270,"13,426.00"
ADJUST,Adjustment by john on 26/01/2010 16,36,"8,897.93"
BANK CHARGES,Bank Charges,34,519.24
GIFT_0001_20,Dotcomgiftshop Gift Voucher £20.00,29,471.06
GIFT_0001_30,Dotcomgiftshop Gift Voucher £30.00,29,610.09
PADS,PADS TO MATCH ALL CUSHIONS,18,0.02


In [9]:
n_nonprod = non_product_mask.sum()
sales = sales[~non_product_mask]
record("Removed non-product rows", sales, f"{n_nonprod:,} postage/fee/adjustment lines")

Removed non-product rows                         rows=1,009,302  cols=8


## 6. Remove zero / negative prices and quantities

With cancellations gone, any remaining zero or negative quantities are stock write-offs ("damaged",
"missing", "check"). They all have a price of zero and no customer attached. Zero-price lines aren't
sales either, so both go.

In [10]:
bad_value = (sales["Price"] <= 0) | (sales["Quantity"] <= 0)
print(sales.loc[bad_value & (sales.Quantity <= 0), "Description"].value_counts().head(8))

n_bad = bad_value.sum()
sales = sales[~bad_value]
record("Removed zero/negative price or quantity", sales, f"{n_bad:,} write-off or zero-price lines")

Description
check                    121
damages                   83
?                         81
damaged                   78
missing                   27
sold as set on dotcom     20
Damaged                   17
smashed                    9
Name: count, dtype: Int64
Removed zero/negative price or quantity          rows=1,003,340  cols=8


## 7. How much revenue do cancellations cost?

Here I only count cancellations of actual products, so a refunded postage charge doesn't inflate
the figure.

In [11]:
product_cancels = cancellations[~is_non_product(cancellations["StockCode"])]
gross_product_sales = (sales["Quantity"] * sales["Price"]).sum()
cancelled_value = -product_cancels["Revenue"].sum()

cancel_summary = pd.Series({
    "Gross product sales (£)": gross_product_sales,
    "Cancelled product value (£)": cancelled_value,
    "Cancelled as % of gross": cancelled_value / gross_product_sales * 100,
    "Cancelled invoices": product_cancels["Invoice"].nunique(),
    "Customers with at least one cancellation": product_cancels["CustomerID"].nunique(),
})
cancel_summary.to_frame("Value")

,Value
Gross product sales (£),"19,643,861.62"
Cancelled product value (£),"716,425.97"
Cancelled as % of gross,3.65
Cancelled invoices,"7,405.00"
Customers with at least one cancellation,"2,444.00"


In [12]:
# Which products get cancelled most (by value)?
(product_cancels.assign(Value=-product_cancels.Revenue)
   .groupby("Description")["Value"].sum()
   .sort_values(ascending=False).head(10)
   .to_frame("Cancelled value (£)"))

,Cancelled value (£)
Description,
"PAPER CRAFT , LITTLE BIRDIE","168,469.60"
MEDIUM CERAMIC TOP STORAGE JAR,"77,479.64"
REGENCY CAKESTAND 3 TIER,"16,545.30"
WHITE HANGING HEART T-LIGHT HOLDER,"9,387.10"
COLOUR GLASS. STAR T-LIGHT HOLDER,"6,966.66"
FAIRY CAKE FLANNEL ASSORTED COLOUR,"6,614.37"
WHITE CHERRY LIGHTS,"6,421.10"
SET/4 WHITE RETRO STORAGE CUBES,"5,187.65"
PANTRY CHOPPING BOARD,"4,803.06"


### Orders that were cancelled outright

Some orders were placed and then reversed in full. The clearest case is a single line for 80,995
units that was cancelled the same day. If I leave the original line in, that customer looks like the
most valuable account in the business when the sale never happened.

To handle this, I match each cancellation line to the most recent earlier order line with the same
customer, product and quantity, and remove the pair. Partial cancellations and anything I can't match
exactly stay in, so this is a conservative fix.

In [13]:
def find_reversed_lines(sales: pd.DataFrame, cancels: pd.DataFrame) -> pd.Index:
    """Return the index of sale lines that a later cancellation exactly reverses.

    A match needs the same customer, stock code and quantity, with the sale dated on or
    before the cancellation. Each cancellation removes at most one sale line (the latest
    one), and each sale line can only be cancelled once.
    """
    c = cancels[cancels["CustomerID"].notna()].copy()
    c["Quantity"] = -c["Quantity"]
    c = c.reset_index(names="cancel_idx")[["cancel_idx", "CustomerID", "StockCode", "Quantity", "InvoiceDate"]]
    s = sales[sales["CustomerID"].notna()].reset_index(names="sale_idx")[
        ["sale_idx", "CustomerID", "StockCode", "Quantity", "InvoiceDate"]]

    pairs = c.merge(s, on=["CustomerID", "StockCode", "Quantity"], suffixes=("_c", "_s"))
    pairs = pairs[pairs["InvoiceDate_s"] <= pairs["InvoiceDate_c"]]
    pairs = pairs.sort_values("InvoiceDate_s").groupby("cancel_idx").tail(1)
    pairs = pairs.drop_duplicates("sale_idx", keep="last")
    return pd.Index(pairs["sale_idx"])


reversed_idx = find_reversed_lines(sales, product_cancels)
reversed_value = (sales.loc[reversed_idx, "Quantity"] * sales.loc[reversed_idx, "Price"]).sum()
print(f"Order lines fully reversed by a cancellation: {len(reversed_idx):,} (£{reversed_value:,.0f})")

# The five biggest reversed lines, to sanity-check the matching
(sales.loc[reversed_idx]
   .assign(Revenue=lambda d: d.Quantity * d.Price)
   .nlargest(5, "Revenue")[["Invoice", "CustomerID", "Description", "Quantity", "Revenue", "InvoiceDate"]])

Order lines fully reversed by a cancellation: 6,180 (£601,523)


,Invoice,CustomerID,Description,Quantity,Revenue,InvoiceDate
sale_idx,,,,,,
1031554,581483,16446,"PAPER CRAFT , LITTLE BIRDIE",80995,"168,469.60",2011-12-09 09:15:00
557400,541431,12346,MEDIUM CERAMIC TOP STORAGE JAR,74215,"77,183.60",2011-01-18 10:01:00
717210,556444,15098,PICNIC BASKET WICKER 60 PIECES,60,"38,970.00",2011-06-10 15:28:00
426971,530715,15838,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,"15,818.40",2010-11-04 11:36:00
548563,540815,15749,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,"6,539.40",2011-01-11 12:55:00


In [14]:
sales = sales.drop(index=reversed_idx)
record("Removed order lines reversed by a cancellation", sales,
       f"{len(reversed_idx):,} lines, £{reversed_value:,.0f}")

Removed order lines reversed by a cancellation   rows=  997,160  cols=8


## 8. Feature columns

In [15]:
sales["Revenue"] = sales["Quantity"] * sales["Price"]
sales["Year"] = sales["InvoiceDate"].dt.year
sales["Month"] = sales["InvoiceDate"].dt.month
sales["YearMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour

record("Added Revenue, Year, Month, Weekday, Hour", sales)
sales.head()

Added Revenue, Year, Month, Weekday, Hour        rows=  997,160  cols=14


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue,Year,Month,YearMonth,Weekday,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,2009,12,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009,12,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009,12,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,2009,12,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,2009,12,2009-12,Tuesday,7


## 9. Split registered customers from guest orders

Rows with no Customer ID can't be tied to a person, so they're useless for RFM or cohorts. They're
still real revenue though, so they get saved to their own file and the EDA notebook adds them back in.

In [16]:
guest_orders = sales[sales["CustomerID"].isna()].copy()
clean = sales[sales["CustomerID"].notna()].copy()

record("Registered-customer rows (clean.csv)", clean)
print(f"Guest-order rows: {len(guest_orders):,}  revenue £{guest_orders.Revenue.sum():,.0f} "
      f"({guest_orders.Revenue.sum() / sales.Revenue.sum():.1%} of total)")

Registered-customer rows (clean.csv)             rows=  770,399  cols=14
Guest-order rows: 226,761  revenue £2,575,279 (13.5% of total)


In [17]:
clean.to_csv(OUT_DIR / "clean.csv", index=False)
guest_orders.to_csv(OUT_DIR / "guest_orders.csv", index=False)
product_cancels.to_csv(OUT_DIR / "cancellations.csv", index=False)

for f in ["clean.csv", "guest_orders.csv", "cancellations.csv"]:
    print(f"{f:<20} {(OUT_DIR / f).stat().st_size / 1e6:6.1f} MB")

clean.csv              95.0 MB
guest_orders.csv       26.6 MB
cancellations.csv       1.8 MB


## 10. Data-quality summary

In [18]:
summary = pd.DataFrame(step_log)
summary["Rows removed"] = (-summary["Rows"].diff()).fillna(0).astype(int).clip(lower=0)
summary["% of raw rows left"] = (summary["Rows"] / summary.loc[0, "Rows"] * 100).round(1)
summary.style.format({"Rows": "{:,}", "Rows removed": "{:,}", "% of raw rows left": "{:.1f}%"}).hide(axis="index")

Step,Rows,Columns,Note,Rows removed,% of raw rows left
Combined both sheets,"1,067,371",8,,0,100.0%
Dropped exact duplicate rows,"1,033,036",8,"34,335 duplicates removed","34,335",96.8%
Fixed dtypes,"1,033,036",8,,0,96.8%
Removed cancellations and A-adjustments,"1,013,926",8,"19,104 cancellation lines kept aside, 6 adjustments dropped","19,110",95.0%
Removed non-product rows,"1,009,302",8,"4,624 postage/fee/adjustment lines","4,624",94.6%
Removed zero/negative price or quantity,"1,003,340",8,"5,962 write-off or zero-price lines","5,962",94.0%
Removed order lines reversed by a cancellation,"997,160",8,"6,180 lines, £601,523","6,180",93.4%
"Added Revenue, Year, Month, Weekday, Hour","997,160",14,,0,93.4%
Registered-customer rows (clean.csv),"770,399",14,,"226,761",72.2%


In [19]:
quality = pd.Series({
    "Date range": f"{clean.InvoiceDate.min():%d %b %Y} to {clean.InvoiceDate.max():%d %b %Y}",
    "Registered customers": f"{clean.CustomerID.nunique():,}",
    "Invoices (registered)": f"{clean.Invoice.nunique():,}",
    "Invoices (guest)": f"{guest_orders.Invoice.nunique():,}",
    "Products": f"{clean.StockCode.nunique():,}",
    "Countries": f"{clean.Country.nunique()}",
    "Revenue, registered customers": f"£{clean.Revenue.sum():,.0f}",
    "Revenue, guest orders": f"£{guest_orders.Revenue.sum():,.0f}",
    "Cancelled product value": f"£{cancelled_value:,.0f}",
})
quality.to_frame("Value")

,Value
Date range,01 Dec 2009 to 09 Dec 2011
Registered customers,"5,839"
Invoices (registered),"36,329"
Invoices (guest),"2,922"
Products,"4,612"
Countries,41
"Revenue, registered customers","£16,467,060"
"Revenue, guest orders","£2,575,279"
Cancelled product value,"£716,426"


## Key Takeaways

- The raw file has **1,067,371 rows**. After cleaning, **997,160 sale lines** are left (93.4%).
- **34,335 rows were exact duplicates.** 22,844 of them come from the 1-9 Dec 2010 overlap between the two sheets, so stacking the sheets without deduplicating would have double-counted that week.
- **Cancellations cost £716,426**, or **3.65% of gross product sales**, across 7,405 cancelled invoices. Two single-line orders (80,995 paper birdies and 74,215 storage jars) make up £246k of that by themselves.
- **6,180 order lines (£601,523) were cancelled in full** and removed together with their cancellation. Otherwise those customers would have looked far more valuable than they are.
- **4,624 lines were postage, fees or adjustments**, and another **5,962 were stock write-offs or zero-price lines**. None of those are product sales.
- **Guest orders (no Customer ID) bring in £2.58M, or 13.5% of revenue.** They count towards the revenue trend but can't be used for RFM or cohorts.
- The customer-level dataset has **5,839 registered customers, 36,329 orders and £16.47M of revenue**.